In [2]:
!pip -q install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [3]:
import random
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate

*seed*

In [4]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

### Load SAMSum dataset

In [6]:
samsum = load_dataset("knkarthick/samsum")
print(samsum)

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


_Make smaller splits for easier fine-tuning_

In [7]:
train_raw = samsum["train"].shuffle(seed=42).select(range(4000))
val_raw = samsum["validation"].shuffle(seed=42).select(range(500))
test_raw = samsum["test"].shuffle(seed=42).select(range(500))

print(len(train_raw), len(val_raw), len(test_raw))

4000 500 500


_Show one raw sample_

In [8]:
sample = train_raw[0]

print("Dialogue:\n")
print(sample["dialogue"])
print("\n" + "=" * 80 + "\n")
print("Reference Summary:\n")
print(sample["summary"])

Dialogue:

Adam: <file_video>
Adam: what do you think
Hector: give me a sec
Hector: ok watching
Adam: let me know
Hector: can't really hear a lot there ;/
Adam: yeah ;/
Adam: i think i need to record it somehow else
Adam: maybe through the interface and software
Hector: that definitely is a great idea!
Hector: i guess that's why i gave you the interface and installed it :D
Adam: yeah xd
Adam: ok i'll try to figure it out later
Hector: ok
Hector: i'll be waiting :P


Reference Summary:

Adam will record it somewhere else through the interface and software. Hector gave and installed the interface before.
